In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC

print('All assets are loaded successfully!')

All assets are loaded successfully!


In [4]:
df = pd.read_csv('training.1600000.processed.noemoticon.csv', encoding='latin-1', header=None)
df = df[[0, 5]]
df.columns = ['sentimental_polarity', 'text']
print(df.head())

   sentimental_polarity                                               text
0                     0  @switchfoot http://twitpic.com/2y1zl - Awww, t...
1                     0  is upset that he can't update his Facebook by ...
2                     0  @Kenichan I dived many times for the ball. Man...
3                     0    my whole body feels itchy and like its on fire 
4                     0  @nationwideclass no, it's not behaving at all....


In [23]:
# remove neutral tweets
df = df[df['sentimental_polarity'] != 2]

# map sentimental_polarity to binary labels (0 -> 0, 4 -> 1)
df['sentimental_polarity'] = df['sentimental_polarity'].map({0: 0, 4: 1})

# print the distribution of mapped labels
print(df['sentimental_polarity'].value_counts())

sentimental_polarity
0    800000
1    800000
Name: count, dtype: int64


In [7]:
#cleaning the tweets
def clean_text(text):
    return text.lower()

df['cleaned_text'] = df['text'].apply(clean_text)

#printing the cleaned text
print('cleaned text:')
print(df['cleaned_text'].head())

print('orignal text:')
print(df['text'].head())


cleaned text:
0    @switchfoot http://twitpic.com/2y1zl - awww, t...
1    is upset that he can't update his facebook by ...
2    @kenichan i dived many times for the ball. man...
3      my whole body feels itchy and like its on fire 
4    @nationwideclass no, it's not behaving at all....
Name: cleaned_text, dtype: object
orignal text:
0    @switchfoot http://twitpic.com/2y1zl - Awww, t...
1    is upset that he can't update his Facebook by ...
2    @Kenichan I dived many times for the ball. Man...
3      my whole body feels itchy and like its on fire 
4    @nationwideclass no, it's not behaving at all....
Name: text, dtype: object


In [8]:
#train test split the cleanded text and labels
x_train, x_test, y_train, y_test = train_test_split(
    df['cleaned_text'],
    df['sentimental_polarity'],
    test_size=0.2,
    random_state=42
)

print('printing the shapes of the train and test sets')
print('x_train shape:', x_train.shape)  
print('y_train shape:', y_train.shape)

printing the shapes of the train and test sets
x_train shape: (1280000,)
y_train shape: (1280000,)


In [10]:
#vectorizing the text using TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

x_train_vectorized = vectorizer.fit_transform(x_train)
x_test_vectorized = vectorizer.transform(x_test)

print('x_train_vectorized shape:', x_train_vectorized.shape)
print('x_test_vectorized shape:', x_test_vectorized.shape)


x_train_vectorized shape: (1280000, 5000)
x_test_vectorized shape: (320000, 5000)


In [13]:
#training a bernoulli model
bernoulli_model = BernoulliNB()
bernoulli_model.fit(x_train_vectorized, y_train)

#predicting on the test set
bernoulli_prediction = bernoulli_model.predict(x_test_vectorized)

#measuring the accuracy of the bernoulli model
bernoulli_accuracy = accuracy_score(y_test, bernoulli_prediction)
print('Bernoulli Naive Bayes Accuracy:', bernoulli_accuracy*100 , '%')

#classification report for bernoulli model
print('classification report for Bernoulli Naive Bayes:')
bernoulli_report = classification_report(y_test, bernoulli_prediction)
print(bernoulli_report)

Bernoulli Naive Bayes Accuracy: 76.6478125 %
classification report for Bernoulli Naive Bayes:
              precision    recall  f1-score   support

           0       0.77      0.75      0.76    159494
           4       0.76      0.78      0.77    160506

    accuracy                           0.77    320000
   macro avg       0.77      0.77      0.77    320000
weighted avg       0.77      0.77      0.77    320000



In [15]:
#training the svm model with 1000 iterations
svm_model = SVC(kernel='linear', max_iter=1000)

#fitting the svm model
svm_model.fit(x_train_vectorized, y_train)

#predicting on the test set
svm_prediction = svm_model.predict(x_test_vectorized)

#measuring the accuracy of the svm model
svm_accuracy = accuracy_score(y_test, svm_prediction)
print('SVM Accuracy:', svm_accuracy*100 , '%')

#classification report for svm model
print('classification report for svm:')
svm_report = classification_report(y_test, svm_prediction)
print(svm_report)


C:\Users\2004s\AppData\Roaming\Python\Python311\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


SVM Accuracy: 53.5503125 %
classification report for svm:
              precision    recall  f1-score   support

           0       0.64      0.15      0.25    159494
           4       0.52      0.92      0.66    160506

    accuracy                           0.54    320000
   macro avg       0.58      0.53      0.46    320000
weighted avg       0.58      0.54      0.46    320000



In [17]:
from sklearn.svm import LinearSVC

linear_svm_model = LinearSVC(max_iter=1000)
linear_svm_model.fit(x_train_vectorized, y_train)
linear_svm_prediction = linear_svm_model.predict(x_test_vectorized)
linear_svm_accuracy = accuracy_score(y_test, linear_svm_prediction)
print('Linear SVM Accuracy:', linear_svm_accuracy*100 , '%')

#classification report for linear svm model
print('classification report for Linear SVM:')
linear_svm_report = classification_report(y_test, linear_svm_prediction)
print(linear_svm_report)

Linear SVM Accuracy: 79.52906250000001 %
classification report for Linear SVM:
              precision    recall  f1-score   support

           0       0.80      0.78      0.79    159494
           4       0.79      0.81      0.80    160506

    accuracy                           0.80    320000
   macro avg       0.80      0.80      0.80    320000
weighted avg       0.80      0.80      0.80    320000



In [18]:
#training on logistic regression
from sklearn.linear_model import LogisticRegression

logistic_regression_model = LogisticRegression(max_iter=1000)

#fitting the model
logistic_regression_model.fit(x_train_vectorized, y_train)

#predicting on the test set
logistic_regression_prediction = logistic_regression_model.predict(x_test_vectorized)

#meassuring the accuracy
accuracy_on_logistic_regression = accuracy_score(y_test, logistic_regression_prediction)
print('Logistic Regression Accuracy:', accuracy_on_logistic_regression*100 , '%')

#classification report for logistic regression
classification_Report_for_logistic_regression = classification_report(y_test, logistic_regression_prediction)
print('classification report for Logistic Regression:')
print(classification_Report_for_logistic_regression)


Logistic Regression Accuracy: 79.6296875 %
classification report for Logistic Regression:
              precision    recall  f1-score   support

           0       0.80      0.78      0.79    159494
           4       0.79      0.81      0.80    160506

    accuracy                           0.80    320000
   macro avg       0.80      0.80      0.80    320000
weighted avg       0.80      0.80      0.80    320000



In [24]:
# prediction on sample tweets
sample_tweets = [
    "I love this product! It's amazing.",
    "This is the worst experience I've ever had.",
    "I'm not sure how I feel about this.",
    "Absolutely fantastic! Highly recommend it.",
    "Terrible service, will never come back."
]
sample_tweets_vectorized = vectorizer.transform(sample_tweets)

# predict sentiment using the logistic regression model
sample_prediction_logistic_model = logistic_regression_model.predict(sample_tweets_vectorized)

# ensure output labels are binary (1 for positive, 0 for negative)
sample_prediction_binary = [1 if label in [1, 4] else 0 for label in sample_prediction_logistic_model]
mapped_sample_prediction_logistic_model = ['positive' if label == 1 else 'negative' for label in sample_prediction_binary]

print('Predicted sentiment for sample tweets using Logistic Regression (binary labels):')
print(sample_prediction_binary)
print(mapped_sample_prediction_logistic_model)

Predicted sentiment for sample tweets using Logistic Regression (binary labels):
[1, 0, 0, 1, 0]
['positive', 'negative', 'negative', 'positive', 'negative']


In [28]:
personal_sample = ['i was in love with you, and what did you do? nothing you left me there all alone, i wanna leave , i did leave, but somehow a pat of me is still stranded there and that parts remonids me everytim that nomatter what happens you will always bleed my heart out']
personal_vect = vectorizer.transform(personal_sample)

#predicting on naive bayes model
prediction = bernoulli_model.predict(personal_vect)

if prediction[0] == 1:
    print('The predicted sentiment for the personal sample is: Positive')
else:
    print('The predicted sentiment for the personal sample is: Negative')
    
print('Predicted sentiment for personal sample using Bernoulli Naive Bayes (binary label):')
print(prediction)


The predicted sentiment for the personal sample is: Negative
Predicted sentiment for personal sample using Bernoulli Naive Bayes (binary label):
[0]
